# Thesis — ESCP Phase 1 & 2 on Google Colab

This notebook clones the repo from GitHub, installs Gurobi, activates a **WLS academic license**, and runs the Phase-1 batch.

**Workflow:** edit code locally → `git push` → re-run the *Get the code* cell here (`git pull`).

**License:** you need a free WLS academic license from https://portal.gurobi.com (Named-user academic licenses do NOT work in the cloud).

## 1. Install Gurobi

In [2]:
!pip install -q gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 68.6 MB/s eta 0:00:00


## 2. Activate the WLS license

Get these three values from your WLS license on the Gurobi portal.

**Tip:** to avoid pasting secrets into the notebook, store them with Colab *Secrets* (the 🔑 icon in the left sidebar) named `WLSACCESSID`, `WLSSECRET`, `LICENSEID`, then use the commented `userdata` block.

In [3]:
# Option A: paste directly (quick, but secrets live in the notebook)
WLSACCESSID= "6a860c34-1310-482c-adb2-d56a29d5398b"
WLSSECRET= "08acddb6-54f4-4e75-810c-fe105f1b6270"
LICENSEID= 2835469

# Option B (recommended): read from Colab Secrets
# from google.colab import userdata
# WLSACCESSID = userdata.get('WLSACCESSID')
# WLSSECRET   = userdata.get('WLSSECRET')
# LICENSEID   = userdata.get('LICENSEID')

# Write a gurobi.lic file so the DEFAULT Gurobi env (used by build_model)
# picks up the WLS credentials automatically — no code changes needed.
import os
lic_path = '/content/gurobi.lic'
with open(lic_path, 'w') as f:
    f.write(f'WLSACCESSID={WLSACCESSID}\n')
    f.write(f'WLSSECRET={WLSSECRET}\n')
    f.write(f'LICENSEID={LICENSEID}\n')
os.environ['GRB_LICENSE_FILE'] = lic_path

# Quick license check (uses the default env)
import gurobipy as gp
_m = gp.Model()
print('Gurobi license OK')

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2835469
Academic license 2835469 - for non-commercial use only - registered to ek___@tum.de
Gurobi license OK


## 3. Get the code (clone or pull)

Re-run this cell whenever you push new commits locally.

In [12]:
import os, sys

REPO_URL = 'https://github.com/ekaterina1tum/Bachelor-Thesis.git'
REPO_DIR = '/content/Bachelor-Thesis'

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

src = os.path.join(REPO_DIR, 'src')
if src not in sys.path:
    sys.path.insert(0, src)

print('src on path:', src)
print(os.listdir(src))

remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 14 (delta 6), reused 12 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (14/14), 6.13 KiB | 1.23 MiB/s, done.
From https://github.com/ekaterina1tum/Bachelor-Thesis
   8446bbb..a034d97  main       -> origin/main
Updating 8446bbb..a034d97
Fast-forward
 src/phase1_export.py   | 202 +++++++++++++++++++++++++++++++++++++++++++++++++
 src/phase2_instance.py |  91 ++++++++++++++++++++++
 2 files changed, 293 insertions(+)
 create mode 100644 src/phase1_export.py
src on path: /content/Bachelor-Thesis/src
['phase2_model.py', 'phase2_example.py', '__pycache__', 'instance.py', 'phase1_batch.py', 'phase2_instance.py', 'phase1_export.py', 'graph.py', 'verify.py', 'model.py']


## 4. Smoke test: solve one instance

In [14]:
from phase1_export import export_instance

DATA = os.path.join(REPO_DIR, 'data', 'MSCDPinstances', '025')

# solves 025_C101, prints obj/F', and writes the solution JSON
doc = export_instance(
    os.path.join(DATA, '025_C101.txt'),
    time_limit=3600,
    max_shift=480,
    out_dir=None,   # -> data/phase1_solutions/025/025_C101.json
)

print('Objective:', doc['objective'])
print("F':", doc['F_prime'])
print('Saved trips:', len(doc['trips']), 'routes:', len(doc['routes']))

025_C101: obj=10862.04  F'=567.04  status=OPTIMAL  ->  /content/Bachelor-Thesis/data/phase1_solutions/025/025_C101.json
Objective: 10862.037546955691
F': 567.0375469556911
Saved trips: 25 routes: 10


In [15]:
from phase1_export import export_instance

for name in ['025_C101.txt', '025_C102.txt', '025_C103.txt']:
    export_instance(os.path.join(DATA, name), 3600, 480, None)

025_C101: obj=10862.04  F'=567.04  status=OPTIMAL  ->  /content/Bachelor-Thesis/data/phase1_solutions/025/025_C101.json
025_C102: obj=7445.74  F'=703.74  status=OPTIMAL  ->  /content/Bachelor-Thesis/data/phase1_solutions/025/025_C102.json
025_C103: obj=6078.69  F'=910.69  status=OPTIMAL  ->  /content/Bachelor-Thesis/data/phase1_solutions/025/025_C103.json


## 5. Run the Phase-1 batch (one type at a time)

Args: `<folder> <time_limit_s> <max_shift> <type_filter>`. Type filter is one of `C1 C2 R1 R2 RC1 RC2`. Omit the filter to run all 56 instances.

In [7]:
!cd {src} && python phase1_batch.py {DATA} 3600 480 C1

Folder: /content/Bachelor-Thesis/data/MSCDPinstances/025
Instances: 9 | time limit: 3600.0s | max_shift: 480.0 | filter: C1

instance        type         F_prime           obj  status    gap%  time(s)
---------------------------------------------------------------------------
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2835469
Academic license 2835469 - for non-commercial use only - registered to ek___@tum.de
025_C101.txt    C1            567.04      10862.04     OPT    0.00      0.2
025_C102.txt    C1            703.74       7445.74     OPT    0.00     20.1
025_C103.txt    C1            910.69       6078.69     OPT    0.00    236.1

Interrupt request received
^C


In [8]:
!cd {src} && python phase1_batch.py {DATA} 3600 480 C2

Folder: /content/Bachelor-Thesis/data/MSCDPinstances/025
Instances: 8 | time limit: 3600.0s | max_shift: 480.0 | filter: C2

instance        type         F_prime           obj  status    gap%  time(s)
---------------------------------------------------------------------------
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2835469
Academic license 2835469 - for non-commercial use only - registered to ek___@tum.de
025_C201.txt    C2            637.52      40029.52     OPT    0.00      0.2
025_C202.txt    C2            851.63      32352.63     OPT    0.00     11.1
025_C203.txt    C2           1066.51      24902.51     OPT    0.00     88.0

Interrupt request received
^C


In [10]:
!cd {src} && python phase1_batch.py {DATA} 3600 480 R1

Folder: /content/Bachelor-Thesis/data/MSCDPinstances/025
Instances: 12 | time limit: 3600.0s | max_shift: 480.0 | filter: R1

instance        type         F_prime           obj  status    gap%  time(s)
---------------------------------------------------------------------------
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2835469
Academic license 2835469 - for non-commercial use only - registered to ek___@tum.de

Interrupt request received
^C
